# 08 - Model Comparison & Evaluation
MAE, MSE, RMSE, MAPE, R², cross-validation, residual analysis, and feature importance / explainability for the best model.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
%matplotlib inline


In [2]:
import joblib
import pandas as pd
from src import config
from src.train import get_feature_matrix, train_test_split_data
from src.evaluate import evaluate_models, cross_validate_model, get_best_model
from src.advanced_analytics import compute_permutation_importance

df = pd.read_csv(config.FEATURED_DATA_FILE, parse_dates=[config.DATETIME_COL])
X, y, feature_cols = get_feature_matrix(df)
X_train, X_test, y_train, y_test = train_test_split_data(X, y)

model_names = ['LinearRegression','DecisionTree','RandomForest','GradientBoosting','SVR','XGBoost','LightGBM']
fitted = {n: joblib.load(config.MODELS_DIR / f'{n}.pkl') for n in model_names if (config.MODELS_DIR / f'{n}.pkl').exists()}
list(fitted.keys())

['LinearRegression',
 'DecisionTree',
 'RandomForest',
 'GradientBoosting',
 'SVR',
 'XGBoost',
 'LightGBM']

## Comparison table (sorted by RMSE)

In [3]:
results = evaluate_models(fitted, X_test, y_test)
results

2026-07-03 05:39:30 | src.evaluate | INFO | Model comparison:
 Rank            Model   MAE     MSE   RMSE   MAPE    R2
    1 GradientBoosting 2.041   8.021  2.832 22.728 0.933
    2         LightGBM 2.066   8.050  2.837 23.309 0.933
    3          XGBoost 2.097   8.193  2.862 23.682 0.932
    4     RandomForest 2.091   8.355  2.890 23.475 0.930
    5 LinearRegression 2.091   8.575  2.928 23.190 0.928
    6     DecisionTree 2.411  12.051  3.471 27.216 0.899
    7              SVR 8.395 105.124 10.253 94.035 0.123


,Rank,Model,MAE,MSE,RMSE,MAPE,R2
0,1,GradientBoosting,2.040706,8.021384,2.832205,22.728083,0.933104
1,2,LightGBM,2.066021,8.050004,2.837253,23.309416,0.932865
2,3,XGBoost,2.096534,8.192845,2.862315,23.681725,0.931674
3,4,RandomForest,2.091158,8.354583,2.890429,23.474798,0.930325
4,5,LinearRegression,2.091104,8.574516,2.928227,23.189766,0.928491
5,6,DecisionTree,2.411228,12.050983,3.471453,27.215819,0.899498
6,7,SVR,8.394975,105.124343,10.253016,94.034644,0.123291


## Best model + 5-fold cross-validation

In [4]:
best_name, best_model = get_best_model(results, fitted)
cv = cross_validate_model(best_model, X, y)
print(f'Best model: {best_name}')
print(f"CV RMSE: {cv['mean']:.3f} +/- {cv['std']:.3f}")

Best model: GradientBoosting
CV RMSE: 2.922 +/- 0.135


## Feature importance (best model)

In [5]:
if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
    display(imp.head(15))
else:
    print('Model has no native feature_importances_; see permutation importance below.')

temperature_celsius_rollmean_14    0.813922
temperature_celsius_rollmean_7     0.126959
uv_index                           0.020295
temperature_celsius_rollmean_3     0.012600
day_of_year                        0.003492
latitude                           0.002775
temperature_celsius_lag_1          0.002660
humidity_index                     0.002196
temperature_celsius_rollstd_14     0.001916
temperature_celsius_rollstd_7      0.001280
pressure_mb                        0.000919
temperature_celsius_lag_7          0.000806
temperature_celsius_lag_2          0.000781
air_quality_Nitrogen_dioxide       0.000716
air_quality_PM2.5                  0.000711
dtype: float64

## Permutation importance

In [6]:
perm_imp = compute_permutation_importance(best_model, X_test, y_test, n_repeats=5)
perm_imp.head(15)

temperature_celsius_rollmean_14    1.295700
temperature_celsius_lag_1          0.062161
uv_index                           0.042855
temperature_celsius_rollmean_7     0.026349
humidity_index                     0.011184
latitude                           0.009146
temperature_celsius_lag_2          0.005892
humidity                           0.005600
day_of_year                        0.004902
wind_kph                           0.004162
temperature_celsius_lag_3          0.003201
wind_pressure_interaction          0.002960
temperature_celsius_rollstd_14     0.002469
temperature_celsius_rollmean_3     0.001729
visibility_miles                   0.001596
dtype: float64